# Redshift distribution bake-off: three layouts, three queries, one defensible decision

Your team has been arguing about Redshift distribution for a week. Today you settle it with three layouts, three queries, three scan-byte measurements. The lab will not pick your distribution for you; it will give you the numbers and the constraint scenario, and you will defend the decision in the reflection cell.

The brief's literal scenario is a 200-million-row order fact. That seed is COPY-impractical for a 90-minute lab window, so this notebook uses 333,000 rows per layout (about 1 million across the three fact copies) and 12 rows for the dimension. The teaching is the pattern (DISTSTYLE plus SORTKEY combinations) and the measurement helper, not the absolute row count. The same DDL and the same comparison motion hold at 200 million rows; the only adjustment is wall-clock.

The four deliverables map to four checkpoints:

1. A single-node `dc2.large` Redshift cluster in the default VPC's public subnet, IP-locked SG, the COPY role attached, the 2-hour safety-net Lambda plus EventBridge schedule both `Enabled`, and the cluster registered as critical in `CLEANUP_MANIFEST` BEFORE the waiter polls. A timestamp side-table proves the ordering.
2. Three tables stood up and loaded: `orders_fact_a` with `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`, `orders_fact_b` with `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)`, and `tier_dim_c` with `DISTSTYLE ALL`. Row counts: 333,000 each for `_a` and `_b`, 12 for `_c`.
3. Nine measurements: three query shapes (high-cardinality join, wide-scan aggregate, single-row lookup) against three layouts. Per-pair `scanned_bytes` and `wall_ms` from `STL_QUERY` plus `STL_SCAN`. Report written to `s3://.../report/measurements.json`. Sanity bounds: KEY+SORTKEY scan_bytes is less than EVEN+SORTKEY scan_bytes on the join query; ALL on the dim wins the lookup query.
4. A `report/decision.md` written to S3 with H2 sections "Layouts measured", "Constraint scenario", "Chosen layout", "Justification". The validator captures the chosen layout verbatim; it does not score the choice.

**Critical resource warning.** This is the Data Engineering track's hourly-bill lab. Redshift `dc2.large` bills $0.25 per hour on demand: $6 per day, $42 per week if you forget about it. A typical session is well under an hour, so the happy-path cost is $0.30 to $0.50. The lab is hard-capped at two hours wall clock; the safety-net Lambda destroys the cluster whether you finished or not. Worst case (kernel died, atexit ran late, safety net fired) is about $0.50. Three independent failures (cleanup skipped, atexit failed, safety-net Lambda failed) would have to compound to leave the cluster running overnight. Set the $20 per month billing alert in your AWS console before you start. Do not close the notebook tab without running cleanup.

**Time.** About 75 minutes hands-on. The cluster waiter (4 to 6 minutes) and the COPY commands (about 90 seconds across all three layouts) dominate wall-clock; the SQL is short.

**Cost.** Redshift hourly is the boss line item. About $0.30 at 75 minutes, $0.50 if the safety net is what saved you. The seed Parquet adds about a nickel. Everything else is free. Session range: $0.50 to $1.50.

This lab maps to the Data Engineering track, Sub-track B: Warehouse and Lakehouse Mastery (lab B3). The senior-DE interview talking point: "I benchmarked KEY+SORTKEY vs EVEN+SORTKEY vs ALL on an N-row fact and picked based on the analyst's 50x daily wide-scan workload."

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0 pyarrow

In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12. This lab is a critical-resource lab; the safety-net Lambda
# plus EventBridge schedule rule are declared here so the cluster cannot
# outlive the 2-hour wall-clock window.
#
# Pragmatic deviations from the brief, called out up front:
#   - 200M-row aspiration is documented; the seed is 333K rows per fact
#     layout to fit the 90-minute time budget.
#   - Measurement helper reads STL_QUERY plus STL_SCAN (provisioned
#     dc2.large exposes both system views). STL_QUERY_METRICS exists too
#     but the STL_QUERY + STL_SCAN join is the simpler shape and is what
#     the cert pattern surfaces.

import atexit
import getpass
import io
import json
import random
import string
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "redshift-perf-bakeoff"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names. Bucket name appends the account ID for global uniqueness.
CLUSTER_ID = f"labrun-{LAB_ID}"
SG_NAME = f"labrun-{LAB_ID}-sg"
COPY_ROLE_NAME = f"labrun-{LAB_ID}-copy-role"
SAFETY_NET_LAMBDA_NAME = f"labrun-{LAB_ID}-safety-net"
SAFETY_NET_RULE_NAME = f"labrun-{LAB_ID}-safety-net-rule"
SAFETY_NET_ROLE_NAME = f"labrun-{LAB_ID}-safety-net-role"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# Cluster shape. dc2.large is the cheapest provisioned node type ($0.25/hour).
NODE_TYPE = "dc2.large"
NUMBER_OF_NODES = 1
DB_NAME = "labrun"
DB_USER = "labrun_admin"
DB_PORT = 5439

# Generated admin password. Redshift requires uppercase, lowercase, and a
# digit. Held in memory only; the redshift-data API uses temporary creds
# via DbUser, not this password.
_pw_chars = string.ascii_letters + string.digits
_rng = random.Random(0xBA1EE0FF)
DB_PASSWORD = "Lab" + "".join(_rng.choice(_pw_chars) for _ in range(20)) + "7"

# Seed config. Deterministic so row counts and Parquet bytes are stable.
# 333K orders per layout times two fact layouts plus 12 tier rows. Small
# enough that all three COPY calls finish in under two minutes total;
# large enough to exercise distkey and sortkey behavior on the scan
# metrics in STL_SCAN.
SEED = 7
ORDERS_ROWS_PER_LAYOUT = 333_000
TIER_ROWS = 12

# Safety-net schedule. The cluster auto-destroys 2 hours after setup runs
# per RESOURCE-SAFETY-SPEC Section 3 Lab 6 mitigation, reused here.
SAFETY_NET_HOURS = 2

# Timestamp side-table for Checkpoint 1. The brief mandates manifest-append
# must precede the waiter loop. These globals are recorded by the manifest
# cell and by the Task 1 cell respectively; Checkpoint 1 compares them.
MANIFEST_APPEND_TS = None  # set in the manifest cell below
WAITER_START_TS = None  # set immediately before redshift.get_waiter(...).wait

# Module-level holders for the measurement run (Task 3) and the decision
# markdown (Task 4). Checkpoint 4 reads these globals.
MEASUREMENTS = []  # filled by Task 3; written to S3 then re-read
DECISION_MARKDOWN = ""  # set by Task 4; written to S3 too

# Three layouts. Names are short and used everywhere downstream.
FACT_A = "orders_fact_a"  # KEY(customer_id) + COMPOUND SORTKEY(order_date)
FACT_B = "orders_fact_b"  # DISTSTYLE EVEN + COMPOUND SORTKEY(order_date)
DIM_C = "tier_dim_c"      # DISTSTYLE ALL on the small dim
LAYOUTS = ["a", "b", "c"]

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Redshift call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2, the cluster and the safety-net pair
# (EventBridge schedule rule plus Lambda) tear down FIRST because they are
# the hourly-billed resources. Standard resources (safety-net role, SG,
# COPY role, S3 bucket) come after. The manifest below lists every
# resource the lab creates in strict reverse-creation, critical-first
# order so the atexit handler tears down the cluster even if the kernel
# dies during the create_cluster waiter.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist. The cluster is one of the
# tagged resources, so an unfinished prior run with a still-billing cluster
# blocks here.
#
# This manifest is built BEFORE Task 1 calls redshift.create_cluster. The
# brief's Checkpoint 1 specifically validates that the cluster manifest
# append precedes the waiter; the MANIFEST_APPEND_TS module global captures
# the wall-clock instant the manifest was finalized so the validator can
# compare it against WAITER_START_TS recorded inside the Task 1 cell.
#
# labrun-checks 0.7.0 ships AWS adapters for redshift_cluster,
# eventbridge_rule, lambda_function, security_group, iam_role, and
# s3_bucket; the default AwsCleanupAdapter handles every entry below.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="redshift_cluster",
        id=CLUSTER_ID,
        region=REGION,
        cli_delete_command=(
            f"aws redshift delete-cluster --cluster-identifier {CLUSTER_ID} "
            f"--skip-final-cluster-snapshot"
        ),
        critical=True,
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=SAFETY_NET_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SAFETY_NET_RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SAFETY_NET_RULE_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_function",
        id=SAFETY_NET_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {SAFETY_NET_LAMBDA_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="iam_role",
        id=COPY_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {COPY_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=SAFETY_NET_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {SAFETY_NET_ROLE_NAME}",
    ),
    CleanupResource(
        type="security_group",
        id=SG_NAME,
        region=REGION,
        cli_delete_command=f"aws ec2 delete-security-group --group-name {SG_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]

# Record the wall-clock instant the manifest was finalized. Checkpoint 1
# compares this against WAITER_START_TS (set inside Task 1) to prove the
# brief's mandated ordering: manifest append precedes the waiter.
MANIFEST_APPEND_TS = time.time()


def _atexit_cleanup():
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes during the create_cluster waiter or
    any other long-running step. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for
    leftover state with the lab's tag and surface the cleanup command
    before creating any new resources. This matters extra for this lab
    because a leftover cluster keeps billing $0.25 per hour.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources")
    print("or, worse, a second Redshift cluster billing in parallel. Run")
    print("the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")
print(f"Manifest finalized at MANIFEST_APPEND_TS={MANIFEST_APPEND_TS:.3f}")

## Task 1: Provision the cluster and the 2-hour safety-net, in that order

This is the hottest cell in the lab. The moment `create_cluster` returns, the meter starts at $0.25 per hour. The order of operations matters: the safety net goes up FIRST so a kernel crash during the cluster waiter still has a kill-switch, then the cluster comes up LAST. The manifest entry for the cluster was already declared in the cell above so atexit catches a kernel crash during the 4-to-6-minute waiter.

Build it in this order:

1. **Create the S3 bucket and the COPY role.** The bucket holds the seed Parquet under `orders_a/`, `orders_b/`, and `tier_c/` plus the measurement report under `report/`. The COPY role carries `s3:GetObject` and `s3:ListBucket` on the bucket; Redshift assumes it for the COPY command.
2. **Create the security group locked to your current public IP.** The cluster lands in the default VPC's public subnet with `PubliclyAccessible=True`. The SG is the only thing keeping the cluster off the public internet.
3. **Create the safety-net IAM role plus Lambda plus EventBridge schedule rule.** The Lambda has one job: at the 2-hour wall-clock mark, delete the cluster but ONLY if the `labrun:lab-slug` tag matches. The tag-scoped delete is the seatbelt that prevents a misconfigured safety net from deleting an unrelated cluster.
4. **Call `redshift.create_cluster`** with the SG attached, the COPY role attached via `IamRoles=[...]`, `PubliclyAccessible=True`, and `--skip-final-cluster-snapshot` (set on `delete_cluster`, which the labrun-checks `redshift_cluster` adapter calls with `SkipFinalClusterSnapshot=True`). The cluster manifest entry is pre-declared; you do not need to append again.
5. **Record `WAITER_START_TS = time.time()` IMMEDIATELY before `waiter.wait(...)`.** Checkpoint 1 reads this and asserts `MANIFEST_APPEND_TS < WAITER_START_TS`.

Waking Redshift up. This is the slow part. About 4 to 6 minutes.

In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, COPY role, security group, safety-net Lambda plus
# EventBridge rule, then create the Redshift cluster and wait for it to
# become available. The order below is deliberate; the cluster is the LAST
# thing created so the safety net is already in place when the meter starts.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift = boto3.client(
    "redshift",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Bucket. us-east-1 rejects LocationConstraint; other regions require it.
# YOUR CODE: s3.create_bucket(Bucket=BUCKET_NAME).
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# 2. COPY role. Trust policy lets redshift.amazonaws.com assume; permission
# policy is read on the lab bucket only.
copy_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "redshift.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
copy_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{BUCKET_NAME}",
            f"arn:aws:s3:::{BUCKET_NAME}/*",
        ],
    }],
}
# YOUR CODE: iam.create_role(
#   RoleName=COPY_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=COPY_ROLE_NAME,
#   PolicyName="labrun-copy-policy",
#   PolicyDocument=json.dumps(copy_inline_policy),
# ).
copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"
print(f"COPY role ARN: {copy_role_arn}")

# 3. Security group locked to the caller's public IP. Port 5439 inbound.
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
my_cidr = f"{my_ip}/32"
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
default_vpc_id = default_vpc["Vpcs"][0]["VpcId"]
sg_resp = ec2.create_security_group(
    GroupName=SG_NAME,
    Description=f"Redshift access for {LAB_ID}, locked to caller IP",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
sg_id = sg_resp["GroupId"]
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[{
        "IpProtocol": "tcp",
        "FromPort": DB_PORT,
        "ToPort": DB_PORT,
        "IpRanges": [{"CidrIp": my_cidr, "Description": "labrun lab caller IP"}],
    }],
)
print(f"Security group created: {sg_id} ({SG_NAME}); ingress 5439 from {my_cidr}")

# 4. Safety-net IAM role + Lambda + EventBridge rule.
sn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
sn_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "redshift:DescribeClusters",
                "redshift:DeleteCluster",
                "redshift:DescribeTags",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   PolicyName="labrun-safety-net-policy",
#   PolicyDocument=json.dumps(sn_inline_policy),
# ).
sn_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{SAFETY_NET_ROLE_NAME}"

# Safety-net Lambda source. Tag-scoped delete: the Lambda confirms the
# cluster's labrun:lab-slug tag matches this lab before calling
# delete-cluster, so a misconfigured safety net cannot delete an unrelated
# cluster. SkipFinalClusterSnapshot=True so delete returns fast.
lambda_source = f'''
import boto3

CLUSTER_ID = "{CLUSTER_ID}"
EXPECTED_TAG_KEY = "{LAB_TAG_KEY}"
EXPECTED_TAG_VALUE = "{LAB_TAG_VALUE}"


def handler(event, context):
    rs = boto3.client("redshift")
    try:
        resp = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)
    except rs.exceptions.ClusterNotFoundFault:
        print(f"Cluster {{CLUSTER_ID}} not found; nothing to delete.")
        return {{"status": "noop"}}
    cluster = resp["Clusters"][0]
    tags = {{t["Key"]: t["Value"] for t in cluster.get("Tags", [])}}
    if tags.get(EXPECTED_TAG_KEY) != EXPECTED_TAG_VALUE:
        print(
            f"Refusing to delete {{CLUSTER_ID}}: tag {{EXPECTED_TAG_KEY}} is "
            f"{{tags.get(EXPECTED_TAG_KEY)!r}}, expected {{EXPECTED_TAG_VALUE!r}}."
        )
        return {{"status": "refused"}}
    rs.delete_cluster(
        ClusterIdentifier=CLUSTER_ID,
        SkipFinalClusterSnapshot=True,
    )
    print(f"Cluster {{CLUSTER_ID}} delete initiated by safety net.")
    return {{"status": "deleted"}}
'''

# Zip the source in memory and create the Lambda function.
zbuf = io.BytesIO()
with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", lambda_source)
zip_bytes = zbuf.getvalue()

# IAM role propagation can take a few seconds before Lambda will accept it.
_lambda_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #   FunctionName=SAFETY_NET_LAMBDA_NAME,
        #   Runtime="python3.11",
        #   Role=sn_role_arn,
        #   Handler="index.handler",
        #   Code={"ZipFile": zip_bytes},
        #   Timeout=60,
        #   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _lambda_deadline:
            time.sleep(5)
            continue
        raise
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{SAFETY_NET_LAMBDA_NAME}"
print(f"Safety-net Lambda created: {lambda_arn}")

# EventBridge schedule rule: fire once at lab_start + SAFETY_NET_HOURS.
# cron(minutes hours day month ? year) for the computed UTC instant.
fire_at = (datetime.now(timezone.utc) + timedelta(hours=SAFETY_NET_HOURS)).replace(microsecond=0)
cron_expr = (
    f"cron({fire_at.minute} {fire_at.hour} "
    f"{fire_at.day} {fire_at.month} ? {fire_at.year})"
)
# YOUR CODE: events.put_rule(
#   Name=SAFETY_NET_RULE_NAME,
#   ScheduleExpression=cron_expr,
#   State="ENABLED",
#   Description=f"Safety net for {LAB_ID}; auto-deletes cluster at +{SAFETY_NET_HOURS}h.",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: lambda_client.add_permission(
#   FunctionName=SAFETY_NET_LAMBDA_NAME,
#   StatementId="labrun-eventbridge-invoke",
#   Action="lambda:InvokeFunction",
#   Principal="events.amazonaws.com",
#   SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
# ).
# YOUR CODE: events.put_targets(
#   Rule=SAFETY_NET_RULE_NAME,
#   Targets=[{"Id": "1", "Arn": lambda_arn}],
# ).
print(f"EventBridge rule {SAFETY_NET_RULE_NAME} scheduled at {cron_expr}")

# 5. Create the Redshift cluster. The manifest entry was pre-declared above
# with critical=True, so atexit catches a kernel crash during the waiter.
# YOUR CODE: redshift.create_cluster(
#   ClusterIdentifier=CLUSTER_ID,
#   NodeType=NODE_TYPE,
#   NumberOfNodes=NUMBER_OF_NODES,
#   ClusterType="single-node",
#   MasterUsername=DB_USER,
#   MasterUserPassword=DB_PASSWORD,
#   DBName=DB_NAME,
#   PubliclyAccessible=True,
#   VpcSecurityGroupIds=[sg_id],
#   IamRoles=[copy_role_arn],
#   Port=DB_PORT,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"create_cluster returned for {CLUSTER_ID}; meter has started at $0.25/hour.")
print("Cluster is in `creating`. Polling once per 15s; about 4 to 6 minutes...")

# Record WAITER_START_TS IMMEDIATELY before the waiter loop. Checkpoint 1
# asserts MANIFEST_APPEND_TS < WAITER_START_TS.
WAITER_START_TS = time.time()

waiter = redshift.get_waiter("cluster_available")
waiter.wait(
    ClusterIdentifier=CLUSTER_ID,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Cluster {CLUSTER_ID} is available.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: cluster is available with the right node shape, the lab SG
# is attached, the safety-net Lambda + EventBridge rule are both Enabled,
# the manifest entry has critical=True, and the manifest append preceded
# the waiter start.

def checkpoint_1(session):
    try:
        redshift_c = boto3.client(
            "redshift",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            resp = redshift_c.describe_clusters(ClusterIdentifier=CLUSTER_ID)
        except ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Redshift cluster {CLUSTER_ID!r} does not exist. "
                        f"Run the Task 1 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        clusters = resp.get("Clusters", [])
        if not clusters:
            return CheckpointResult(
                status="fail",
                error_reason=f"describe_clusters returned no clusters for {CLUSTER_ID!r}.",
            )
        cluster = clusters[0]
        status_ = cluster.get("ClusterStatus", "")
        if status_ != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster status is {status_!r}, expected 'available'. "
                    f"Wait for the create_cluster waiter to finish, then re-run "
                    f"this checkpoint."
                ),
            )
        if cluster.get("NodeType") != NODE_TYPE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NodeType is {cluster.get('NodeType')!r}, expected {NODE_TYPE!r}."
                ),
            )
        if not cluster.get("PubliclyAccessible", False):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "PubliclyAccessible is False. The lab uses the default VPC's "
                    "public subnet with an IP-locked SG; PubliclyAccessible must be True."
                ),
            )

        sg_ids = {sg.get("VpcSecurityGroupId", "") for sg in cluster.get("VpcSecurityGroups", [])}
        ec2_c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        sg_resp = ec2_c.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
        )
        lab_sg_ids = {g["GroupId"] for g in sg_resp.get("SecurityGroups", [])}
        if not (sg_ids & lab_sg_ids):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster VpcSecurityGroupIds {sorted(sg_ids)} does not contain "
                    f"the lab SG (one of {sorted(lab_sg_ids)} for {SG_NAME!r})."
                ),
            )

        # Safety-net Lambda must exist and be Active.
        lambda_c = boto3.client(
            "lambda",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            cfg = lambda_c.get_function_configuration(FunctionName=SAFETY_NET_LAMBDA_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Safety-net Lambda {SAFETY_NET_LAMBDA_NAME!r} does not exist. "
                        f"The 2-hour kill-switch is not in place."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        if cfg.get("State") not in ("Active", None):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Safety-net Lambda state is {cfg.get('State')!r}, expected 'Active'."
                ),
            )

        # EventBridge schedule rule must be ENABLED.
        events_c = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            rule = events_c.describe_rule(Name=SAFETY_NET_RULE_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"EventBridge rule {SAFETY_NET_RULE_NAME!r} does not exist. "
                        f"The 2-hour schedule is not in place."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))
        if rule.get("State") != "ENABLED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventBridge rule state is {rule.get('State')!r}, expected 'ENABLED'."
                ),
            )

        # Manifest contains the cluster entry with critical=True.
        manifest = globals().get("CLEANUP_MANIFEST", [])
        cluster_entry = next(
            (r for r in manifest if r.type == "redshift_cluster" and r.id == CLUSTER_ID),
            None,
        )
        if cluster_entry is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster {CLUSTER_ID!r} is not in CLEANUP_MANIFEST. atexit "
                    f"cannot tear it down on kernel crash."
                ),
            )
        if not getattr(cluster_entry, "critical", False):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster entry in CLEANUP_MANIFEST is missing critical=True. "
                    f"Per RESOURCE-SAFETY-SPEC Rule 2 the cluster tears down first."
                ),
            )

        # Timestamp ordering: manifest append preceded waiter start.
        m_ts = globals().get("MANIFEST_APPEND_TS")
        w_ts = globals().get("WAITER_START_TS")
        if m_ts is None:
            return CheckpointResult(
                status="fail",
                error_reason="MANIFEST_APPEND_TS is None. The manifest cell did not run.",
            )
        if w_ts is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "WAITER_START_TS is None. The Task 1 cell did not record the "
                    "waiter start instant; the brief mandates this for the "
                    "manifest-before-waiter proof."
                ),
            )
        if not (m_ts < w_ts):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"MANIFEST_APPEND_TS={m_ts:.3f} is not less than WAITER_START_TS={w_ts:.3f}. "
                    f"The manifest must be finalized before the cluster waiter polls so "
                    f"atexit can tear down a half-created cluster."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 1 walks seven properties: cluster `available`, `dc2.large`, `PubliclyAccessible=True`, lab SG attached, safety-net Lambda exists, EventBridge rule `ENABLED`, manifest entry has `critical=True`, and `MANIFEST_APPEND_TS < WAITER_START_TS`. Read the failure reason. The two most common misses on the first run: forgetting `VpcSecurityGroupIds=[sg_id]` in `create_cluster` (the cluster lands behind the default SG and the SG check fails); and forgetting to record `WAITER_START_TS = time.time()` immediately before `waiter.wait(...)` (the timestamp ordering check fails because `WAITER_START_TS` stays None).

</details>

<details><summary>Hint 2 (direction)</summary>

Eight API calls in this cell. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1). `iam.create_role(RoleName=COPY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(copy_trust_policy), Tags=[...])` plus `iam.put_role_policy(...)`. Same pair for `SAFETY_NET_ROLE_NAME`. `lambda_client.create_function(FunctionName=SAFETY_NET_LAMBDA_NAME, Runtime='python3.11', Role=sn_role_arn, Handler='index.handler', Code={'ZipFile': zip_bytes}, Timeout=60, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` inside the retry loop. Then the three EventBridge wiring calls: `events.put_rule(Name=..., ScheduleExpression=cron_expr, State='ENABLED', ...)`, `lambda_client.add_permission(...)`, `events.put_targets(Rule=..., Targets=[{'Id': '1', 'Arn': lambda_arn}])`. Finally `redshift.create_cluster(...)`. Record `WAITER_START_TS = time.time()` IMMEDIATELY before `waiter.wait(...)` so the timestamp ordering check passes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)

iam.create_role(
    RoleName=COPY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=COPY_ROLE_NAME,
    PolicyName="labrun-copy-policy",
    PolicyDocument=json.dumps(copy_inline_policy),
)

iam.create_role(
    RoleName=SAFETY_NET_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=SAFETY_NET_ROLE_NAME,
    PolicyName="labrun-safety-net-policy",
    PolicyDocument=json.dumps(sn_inline_policy),
)

# Inside the retry loop:
lambda_client.create_function(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    Runtime="python3.11",
    Role=sn_role_arn,
    Handler="index.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=60,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

events.put_rule(
    Name=SAFETY_NET_RULE_NAME,
    ScheduleExpression=cron_expr,
    State="ENABLED",
    Description=f"Safety net for {LAB_ID}; auto-deletes cluster at +{SAFETY_NET_HOURS}h.",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
lambda_client.add_permission(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    StatementId="labrun-eventbridge-invoke",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
)
events.put_targets(
    Rule=SAFETY_NET_RULE_NAME,
    Targets=[{"Id": "1", "Arn": lambda_arn}],
)

redshift.create_cluster(
    ClusterIdentifier=CLUSTER_ID,
    NodeType=NODE_TYPE,
    NumberOfNodes=NUMBER_OF_NODES,
    ClusterType="single-node",
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    DBName=DB_NAME,
    PubliclyAccessible=True,
    VpcSecurityGroupIds=[sg_id],
    IamRoles=[copy_role_arn],
    Port=DB_PORT,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check. Meter started.** The moment `create_cluster` returned, the Redshift `dc2.large` node started billing at $0.25 per hour. Four to six minutes for the cluster to reach `available` is about 2 to 3 cents. The 2-hour EventBridge safety net is the seatbelt; even if your kernel dies right now, the Lambda will tag-check the cluster and call `delete-cluster` at the 2-hour mark, capping worst case at $0.50. The bucket, two IAM roles, security group, Lambda function, and EventBridge rule are all free at this volume. Keep moving; Tasks 2 through 4 are bytes-cheap but the cluster runs the meter until cleanup deletes it.

## Task 2: Stand up three layouts and load the seed Parquet

Three tables. Three distribution choices. One COPY pattern. The teaching here is not the seed; it is the DISTSTYLE plus SORTKEY combinations and how COPY behaves under each.

- `orders_fact_a` uses `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`. Rows are hashed across slices by customer; joins on customer_id stay local.
- `orders_fact_b` uses `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)`. Rows are round-robin across slices; any join on customer_id forces redistribution.
- `tier_dim_c` uses `DISTSTYLE ALL`. The entire 12-row dim is copied to every slice; lookups never broadcast.

Build it in this order:

1. **Generate the seed Parquet in memory** (pandas + pyarrow). 333,000 rows each for `orders_a` and `orders_b` (identical row content; the distribution lives in the table DDL, not the data), 12 rows for the dim. Upload to `s3://{BUCKET}/orders_a/`, `s3://{BUCKET}/orders_b/`, `s3://{BUCKET}/tier_c/`.
2. **CREATE TABLE three times** with the three distribution clauses.
3. **COPY from S3** for each table using the COPY role attached at cluster create time. COPY is loading 333K rows from S3. About 30 seconds each.

The redshift-data API is the credential-free interface: pass `ClusterIdentifier`, `Database`, `DbUser`; redshift-data uses your IAM identity to mint a temporary database credential and run the SQL.

In [ ]:
# NBVAL_SKIP
# Task 2: Build the seed Parquet, upload to S3, CREATE the three layouts,
# COPY into each. The teaching is the DISTSTYLE + SORTKEY combinations;
# the row content is identical across _a and _b on purpose.

import pyarrow as pa
import pyarrow.parquet as pq

random.seed(SEED)

redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_redshift_sql(sql, with_result=False):
    """Submit SQL via redshift-data, poll until terminal, return describe payload.

    When with_result is True the function also fetches and attaches the
    first page of get_statement_result under key 'result'. Raises
    RuntimeError on FAILED or ABORTED so the cell stops on a bad query.
    """
    start = redshift_data.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=sql,
    )
    sid = start["Id"]
    deadline = time.time() + 180
    while time.time() < deadline:
        desc = redshift_data.describe_statement(Id=sid)
        status_ = desc["Status"]
        if status_ in ("FINISHED", "FAILED", "ABORTED"):
            if status_ != "FINISHED":
                err = desc.get("Error", "(no error message)")
                raise RuntimeError(f"redshift-data {sid} ended {status_}: {err}")
            if with_result and desc.get("HasResultSet"):
                desc["result"] = redshift_data.get_statement_result(Id=sid)
            return desc
        time.sleep(1)
    raise RuntimeError(f"redshift-data {sid} did not finish within 3 minutes.")


# 1. Build seed Parquet. Identical row content for _a and _b; only the table
# DDL differs. The dim is a tiny 12-row tier lookup that will go DISTSTYLE ALL.
START_DATE = datetime(2024, 1, 1)
TIERS = ["bronze", "silver", "gold", "platinum"]
STATUSES = ["pending", "shipped", "delivered", "cancelled"]

orders_data = {
    "order_id": list(range(1, ORDERS_ROWS_PER_LAYOUT + 1)),
    "customer_id": [random.randint(1, 100_000) for _ in range(ORDERS_ROWS_PER_LAYOUT)],
    "order_date": [
        (START_DATE + timedelta(days=random.randint(0, 364))).date().isoformat()
        for _ in range(ORDERS_ROWS_PER_LAYOUT)
    ],
    "amount_cents": [random.randint(100, 99999) for _ in range(ORDERS_ROWS_PER_LAYOUT)],
    "status": [random.choice(STATUSES) for _ in range(ORDERS_ROWS_PER_LAYOUT)],
    "tier_id": [random.randint(0, 11) for _ in range(ORDERS_ROWS_PER_LAYOUT)],
}
orders_table = pa.table(orders_data)
buf = io.BytesIO()
pq.write_table(orders_table, buf)
orders_bytes = buf.getvalue()
print(f"orders Parquet built: {ORDERS_ROWS_PER_LAYOUT} rows, {len(orders_bytes)} bytes")

# Upload the same bytes to both _a and _b prefixes. COPY reads each prefix
# into its own table; the layouts differ in DDL, not in source data.
# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key='orders_a/orders.parquet', Body=orders_bytes).
# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key='orders_b/orders.parquet', Body=orders_bytes).
print(f"orders Parquet uploaded to s3://{BUCKET_NAME}/orders_a/ and orders_b/")

# 12-row tier dim. Small, static, perfect for DISTSTYLE ALL.
tier_data = {
    "tier_id": list(range(12)),
    "tier_name": [random.choice(TIERS) for _ in range(12)],
    "discount_pct": [round(random.uniform(0, 0.3), 3) for _ in range(12)],
}
tier_table = pa.table(tier_data)
buf = io.BytesIO()
pq.write_table(tier_table, buf)
tier_bytes = buf.getvalue()
# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key='tier_c/tier.parquet', Body=tier_bytes).
print(f"tier dim Parquet uploaded ({TIER_ROWS} rows, {len(tier_bytes)} bytes)")

# 2. Create the three layouts. KEY+SORT, EVEN+SORT, ALL.
# YOUR CODE: run_redshift_sql(f"DROP TABLE IF EXISTS {FACT_A};").
# YOUR CODE: run_redshift_sql(
#   f"CREATE TABLE {FACT_A} ("
#   "  order_id BIGINT, customer_id BIGINT, order_date DATE, "
#   "  amount_cents BIGINT, status VARCHAR(16), tier_id INTEGER"
#   ") DISTKEY(customer_id) COMPOUND SORTKEY(order_date);"
# ).
# YOUR CODE: run_redshift_sql(f"DROP TABLE IF EXISTS {FACT_B};").
# YOUR CODE: run_redshift_sql(
#   f"CREATE TABLE {FACT_B} ("
#   "  order_id BIGINT, customer_id BIGINT, order_date DATE, "
#   "  amount_cents BIGINT, status VARCHAR(16), tier_id INTEGER"
#   ") DISTSTYLE EVEN COMPOUND SORTKEY(order_date);"
# ).
# YOUR CODE: run_redshift_sql(f"DROP TABLE IF EXISTS {DIM_C};").
# YOUR CODE: run_redshift_sql(
#   f"CREATE TABLE {DIM_C} ("
#   "  tier_id INTEGER, tier_name VARCHAR(16), discount_pct FLOAT"
#   ") DISTSTYLE ALL;"
# ).
print("Three layouts created in Redshift.")

# 3. COPY each prefix into its layout. COPY is loading 333K rows from S3.
# About 30 seconds each.
copy_template = (
    "COPY {table} FROM 's3://{bucket}/{prefix}/' "
    "IAM_ROLE '{role_arn}' FORMAT AS PARQUET;"
)
for table, prefix in [(FACT_A, "orders_a"), (FACT_B, "orders_b"), (DIM_C, "tier_c")]:
    sql = copy_template.format(
        table=table, bucket=BUCKET_NAME, prefix=prefix, role_arn=copy_role_arn,
    )
    # YOUR CODE: run_redshift_sql(sql).
    print(f"COPY {table} <- s3://{BUCKET_NAME}/{prefix}/ : done.")

# Sanity print row counts so the layout looks right before Checkpoint 2.
for table in [FACT_A, FACT_B, DIM_C]:
    desc = run_redshift_sql(f"SELECT count(*) FROM {table};", with_result=True)
    rows = desc["result"]["Records"]
    count = int(rows[0][0]["longValue"]) if rows else 0
    print(f"{table}: {count} rows")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Three tables exist with the declared distribution and sort
# strategies (read from SVV_TABLE_INFO) and the row counts match the seed.

def checkpoint_2(session):
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run(sql):
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 60
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                if desc["Status"] in ("FINISHED", "FAILED", "ABORTED"):
                    if desc["Status"] != "FINISHED":
                        raise RuntimeError(f"{sid} {desc['Status']}: {desc.get('Error','')}")
                    return rd.get_statement_result(Id=sid) if desc.get("HasResultSet") else None
                time.sleep(1)
            raise RuntimeError(f"{sid} did not finish in 60s")

        # SVV_TABLE_INFO returns diststyle, sortkey1 per table. diststyle values:
        # 'KEY(customer_id)', 'EVEN', 'ALL', etc. sortkey1 is the first sortkey
        # column name.
        meta_sql = (
            "SELECT \"table\", diststyle, sortkey1 FROM SVV_TABLE_INFO "
            f"WHERE \"table\" IN ('{FACT_A}', '{FACT_B}', '{DIM_C}');"
        )
        res = run(meta_sql)
        records = res.get("Records", []) if res else []
        meta = {}
        for row in records:
            name = row[0].get("stringValue", "")
            diststyle = row[1].get("stringValue", "")
            sortkey1 = row[2].get("stringValue", "") if len(row) > 2 else ""
            meta[name] = {"diststyle": diststyle, "sortkey1": sortkey1}

        for t in (FACT_A, FACT_B, DIM_C):
            if t not in meta:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {t} not visible in SVV_TABLE_INFO.",
                )

        a_ds = meta[FACT_A]["diststyle"].upper()
        if not a_ds.startswith("KEY") or "customer_id" not in a_ds.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{FACT_A} diststyle is {meta[FACT_A]['diststyle']!r}; "
                    f"expected KEY(customer_id)."
                ),
            )
        if meta[FACT_A]["sortkey1"].lower() != "order_date":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{FACT_A} first sortkey is {meta[FACT_A]['sortkey1']!r}; "
                    f"expected order_date."
                ),
            )

        b_ds = meta[FACT_B]["diststyle"].upper()
        if b_ds != "EVEN":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{FACT_B} diststyle is {meta[FACT_B]['diststyle']!r}; expected EVEN."
                ),
            )
        if meta[FACT_B]["sortkey1"].lower() != "order_date":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{FACT_B} first sortkey is {meta[FACT_B]['sortkey1']!r}; "
                    f"expected order_date."
                ),
            )

        c_ds = meta[DIM_C]["diststyle"].upper()
        if c_ds != "ALL":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{DIM_C} diststyle is {meta[DIM_C]['diststyle']!r}; expected ALL."
                ),
            )

        # Row counts.
        for t, expected in ((FACT_A, ORDERS_ROWS_PER_LAYOUT), (FACT_B, ORDERS_ROWS_PER_LAYOUT), (DIM_C, TIER_ROWS)):
            res = run(f"SELECT count(*) FROM {t};")
            recs = res.get("Records", []) if res else []
            count = int(recs[0][0]["longValue"]) if recs else 0
            if count != expected:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{t} has {count} rows; expected {expected}. "
                        f"COPY may have failed silently; query STL_LOAD_ERRORS for details."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 2 walks two things: each of the three tables shows the expected `diststyle` and `sortkey1` in `SVV_TABLE_INFO`, and each table has the expected row count. The most common miss is a COPY that reported `FINISHED` but landed zero rows because the IAM role ARN was wrong; `redshift-data` does not raise on a zero-row COPY. Read the failure reason: it names the specific table and tells you whether the issue is the layout DDL or the data load.

</details>

<details><summary>Hint 2 (direction)</summary>

Three `s3.put_object` calls (`orders_a/orders.parquet`, `orders_b/orders.parquet`, `tier_c/tier.parquet`), then three `run_redshift_sql(DROP)` + three `run_redshift_sql(CREATE)` calls with the three distribution clauses (`DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`, `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)`, `DISTSTYLE ALL`), then three `run_redshift_sql(COPY)` calls. The COPY syntax is `COPY {table} FROM 's3://{bucket}/{prefix}/' IAM_ROLE '{role_arn}' FORMAT AS PARQUET;`. The IAM_ROLE ARN must be the COPY role attached at cluster create, not the safety-net role.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
s3.put_object(Bucket=BUCKET_NAME, Key="orders_a/orders.parquet", Body=orders_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="orders_b/orders.parquet", Body=orders_bytes)
s3.put_object(Bucket=BUCKET_NAME, Key="tier_c/tier.parquet", Body=tier_bytes)

run_redshift_sql(f"DROP TABLE IF EXISTS {FACT_A};")
run_redshift_sql(
    f"CREATE TABLE {FACT_A} ("
    "  order_id BIGINT, customer_id BIGINT, order_date DATE, "
    "  amount_cents BIGINT, status VARCHAR(16), tier_id INTEGER"
    ") DISTKEY(customer_id) COMPOUND SORTKEY(order_date);"
)
run_redshift_sql(f"DROP TABLE IF EXISTS {FACT_B};")
run_redshift_sql(
    f"CREATE TABLE {FACT_B} ("
    "  order_id BIGINT, customer_id BIGINT, order_date DATE, "
    "  amount_cents BIGINT, status VARCHAR(16), tier_id INTEGER"
    ") DISTSTYLE EVEN COMPOUND SORTKEY(order_date);"
)
run_redshift_sql(f"DROP TABLE IF EXISTS {DIM_C};")
run_redshift_sql(
    f"CREATE TABLE {DIM_C} ("
    "  tier_id INTEGER, tier_name VARCHAR(16), discount_pct FLOAT"
    ") DISTSTYLE ALL;"
)

for table, prefix in [(FACT_A, "orders_a"), (FACT_B, "orders_b"), (DIM_C, "tier_c")]:
    sql = copy_template.format(
        table=table, bucket=BUCKET_NAME, prefix=prefix, role_arn=copy_role_arn,
    )
    run_redshift_sql(sql)
```

</details>

**Wallet check.** Cluster meter still running at $0.25 per hour. COPY itself is free (Redshift bills for node-hours, not for COPY operations). The three small Parquet uploads are pennies of S3 PUT requests, well inside the always-free request quota. Total session cost so far is dominated by cluster wall-clock since `create_cluster`. About 8 to 10 minutes elapsed at this point, so 3 to 4 cents.

## Task 3: Run the measurement helper across three queries by three layouts

Three query shapes against three layouts: nine measurements. Each measurement captures `scanned_bytes` (sum of `bytes` from `STL_SCAN` rows for the query's scan nodes) and `wall_ms` (the redshift-data `Duration` field, nanoseconds to milliseconds).

The three query shapes:

- **Q_join** (high-cardinality join on customer_id): `SELECT customer_id, count(*) FROM {fact} GROUP BY customer_id ORDER BY count(*) DESC LIMIT 10;`. KEY+SORTKEY wins because customer rows are co-located on a single slice.
- **Q_widescan** (wide-scan aggregate over date range): `SELECT date_trunc('month', order_date), sum(amount_cents) FROM {fact} WHERE order_date BETWEEN '2024-03-01' AND '2024-09-30' GROUP BY 1;`. Sortkey on `order_date` makes both KEY and EVEN scan a similar number of bytes; differences are noise.
- **Q_lookup** (single-row dim lookup): `SELECT * FROM {dim} WHERE tier_id = 7;`. ALL distribution wins because every slice has the dim locally; KEY/EVEN on the fact are not relevant for this shape, so the lookup is measured ONLY against `{DIM_C}`.

Run shape:

- For Q_join and Q_widescan: run the query against `FACT_A`, `FACT_B`. (Two layouts × two queries = four measurements.)
- For Q_lookup: run against `DIM_C` only; the corresponding entries for `FACT_A` and `FACT_B` use a fact-shaped lookup `SELECT * FROM {fact} WHERE order_id = 1234;` so the comparison surface is symmetric. (Three layouts × one query = three measurements.)

Total 4 + 3 + 2 wide-scan = 9 measurements (Q_widescan against `FACT_A`, `FACT_B` and a tier-only `FACT_A` placeholder makes 3 per query × 3 queries = 9). The helper handles the dispatch; you just run it. STL_QUERY is rolling up the latest scan bytes.

In [ ]:
# NBVAL_SKIP
# Task 3: Measurement helper. For each (query_shape, layout) pair, run the
# query, then read STL_QUERY + STL_SCAN to capture scan bytes and wall_ms.
# Write the result list to s3://{BUCKET}/report/measurements.json.

# Three queries x three layouts. The lookup query against the fact uses
# order_id (a low-cardinality field for the small 333K row set); against
# the dim it uses tier_id. The wide-scan against the dim uses a 1:1
# select-all to keep the shape consistent.
QUERY_SHAPES = {
    "q_join": {
        FACT_A: f"SELECT customer_id, count(*) FROM {FACT_A} GROUP BY customer_id ORDER BY count(*) DESC LIMIT 10",
        FACT_B: f"SELECT customer_id, count(*) FROM {FACT_B} GROUP BY customer_id ORDER BY count(*) DESC LIMIT 10",
        DIM_C: f"SELECT tier_id, count(*) FROM {DIM_C} GROUP BY tier_id ORDER BY count(*) DESC LIMIT 10",
    },
    "q_widescan": {
        FACT_A: f"SELECT date_trunc('month', order_date), sum(amount_cents) FROM {FACT_A} WHERE order_date BETWEEN '2024-03-01' AND '2024-09-30' GROUP BY 1",
        FACT_B: f"SELECT date_trunc('month', order_date), sum(amount_cents) FROM {FACT_B} WHERE order_date BETWEEN '2024-03-01' AND '2024-09-30' GROUP BY 1",
        DIM_C: f"SELECT discount_pct, count(*) FROM {DIM_C} GROUP BY discount_pct",
    },
    "q_lookup": {
        FACT_A: f"SELECT * FROM {FACT_A} WHERE order_id = 1234",
        FACT_B: f"SELECT * FROM {FACT_B} WHERE order_id = 1234",
        DIM_C: f"SELECT * FROM {DIM_C} WHERE tier_id = 7",
    },
}


def measure_one(shape, layout, sql):
    """Run one query, then read STL_QUERY + STL_SCAN for scan_bytes + wall_ms."""
    desc = run_redshift_sql(sql)
    duration_ns = desc.get("Duration", 0)
    wall_ms = int(duration_ns / 1_000_000)
    # redshift-data does NOT return the Redshift internal query id directly;
    # we query STL_QUERY by the SQL text fragment to get the most recent
    # query id for this user, then sum STL_SCAN.bytes.
    sql_fragment = sql.replace("'", "''")[:80]
    stl_query_sql = (
        "SELECT query FROM STL_QUERY "
        f"WHERE querytxt LIKE '{sql_fragment}%' "
        "ORDER BY starttime DESC LIMIT 1"
    )
    res = run_redshift_sql(stl_query_sql, with_result=True)
    records = res.get("result", {}).get("Records", [])
    if not records:
        return {
            "shape": shape, "layout": layout,
            "scanned_bytes": 0, "wall_ms": wall_ms,
            "note": "STL_QUERY had no row for this query text",
        }
    query_id = int(records[0][0]["longValue"])
    scan_sql = (
        f"SELECT coalesce(sum(bytes), 0) FROM STL_SCAN WHERE query = {query_id}"
    )
    res = run_redshift_sql(scan_sql, with_result=True)
    recs = res.get("result", {}).get("Records", [])
    scanned_bytes = int(recs[0][0].get("longValue", 0)) if recs else 0
    return {
        "shape": shape,
        "layout": layout,
        "scanned_bytes": scanned_bytes,
        "wall_ms": wall_ms,
        "query_id": query_id,
    }


# Run the matrix.
# YOUR CODE: build MEASUREMENTS by iterating shape/layout pairs and calling
# measure_one(shape, layout, sql). The result list should have 9 entries.
MEASUREMENTS.clear()
for shape, layouts_sql in QUERY_SHAPES.items():
    for layout, sql in layouts_sql.items():
        m = measure_one(shape, layout, sql)
        MEASUREMENTS.append(m)
        print(f"{shape:>11} on {layout:>14}: scan_bytes={m['scanned_bytes']:>10} wall_ms={m['wall_ms']:>5}")

# Persist to S3 so Checkpoint 3 can re-read independently.
report_bytes = json.dumps(MEASUREMENTS, indent=2).encode("utf-8")
# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key='report/measurements.json', Body=report_bytes).
print()
print(f"measurements.json uploaded to s3://{BUCKET_NAME}/report/measurements.json ({len(MEASUREMENTS)} entries)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: The measurement report exists at s3://.../report/measurements.json,
# has 9 entries (3 shapes x 3 layouts), each entry has scanned_bytes and wall_ms,
# and the sanity bounds hold:
#   - q_join: FACT_A scan_bytes < FACT_B scan_bytes (KEY co-located beats EVEN redistribute)
#   - q_lookup: DIM_C scan_bytes < FACT_A scan_bytes and DIM_C scan_bytes < FACT_B scan_bytes
# Tolerances are generous because STL metrics fluctuate.

def checkpoint_3(session):
    try:
        s3_c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_c.get_object(Bucket=BUCKET_NAME, Key="report/measurements.json")
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"s3://{BUCKET_NAME}/report/measurements.json does not exist. "
                        f"Task 3 must upload the measurement report."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        try:
            data = json.loads(obj["Body"].read().decode("utf-8"))
        except Exception as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"measurements.json is not valid JSON: {e}",
            )
        if not isinstance(data, list):
            return CheckpointResult(
                status="fail",
                error_reason="measurements.json must be a JSON list.",
            )
        if len(data) != 9:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"measurements.json has {len(data)} entries; expected 9 "
                    f"(3 query shapes x 3 layouts)."
                ),
            )
        required_keys = {"shape", "layout", "scanned_bytes", "wall_ms"}
        seen = set()
        by_pair = {}
        for entry in data:
            if not required_keys.issubset(entry.keys()):
                missing = required_keys - entry.keys()
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"measurements entry missing keys {sorted(missing)}: {entry}"
                    ),
                )
            pair = (entry["shape"], entry["layout"])
            if pair in seen:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Duplicate (shape, layout) pair in report: {pair}.",
                )
            seen.add(pair)
            by_pair[pair] = entry

        expected_pairs = {
            (s, l) for s in ("q_join", "q_widescan", "q_lookup")
            for l in (FACT_A, FACT_B, DIM_C)
        }
        missing_pairs = expected_pairs - seen
        if missing_pairs:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"measurements.json missing pairs: {sorted(missing_pairs)}."
                ),
            )

        # Sanity bound 1: q_join on FACT_A scans <= FACT_B (KEY co-located
        # beats EVEN redistribute). Tolerance: FACT_A may scan SLIGHTLY more
        # if STL is noisy, but it should not be more than 1.2x FACT_B.
        ja = by_pair[("q_join", FACT_A)]["scanned_bytes"]
        jb = by_pair[("q_join", FACT_B)]["scanned_bytes"]
        if jb > 0 and ja > 1.2 * jb:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"q_join sanity bound violated: FACT_A scanned {ja} bytes, "
                    f"FACT_B scanned {jb}; expected FACT_A <= ~1.2x FACT_B. "
                    f"Did COPY land into the wrong tables?"
                ),
            )

        # Sanity bound 2: q_lookup on DIM_C scans < FACT_A and < FACT_B.
        la = by_pair[("q_lookup", FACT_A)]["scanned_bytes"]
        lb = by_pair[("q_lookup", FACT_B)]["scanned_bytes"]
        lc = by_pair[("q_lookup", DIM_C)]["scanned_bytes"]
        # DIM_C must be strictly less than both fact lookups; if the dim is
        # ALL-distributed, the planner reads only one slice's copy.
        # Tolerance: lc must be <= 50% of min(la, lb) to claim a real win.
        # Allow lc == 0 (single-block read).
        min_fact = min(la, lb) if min(la, lb) > 0 else max(la, lb)
        if min_fact > 0 and lc > 0.5 * min_fact:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"q_lookup sanity bound violated: DIM_C scanned {lc} bytes, "
                    f"min(FACT_A, FACT_B) scanned {min_fact}; expected DIM_C < 0.5x. "
                    f"Is the dim actually DISTSTYLE ALL?"
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 3 walks four things: the JSON exists, it has 9 entries, every entry has the four required keys (`shape`, `layout`, `scanned_bytes`, `wall_ms`), and the two sanity bounds hold (KEY beats EVEN on join, ALL beats both fact layouts on lookup). The most common miss is forgetting to call `s3.put_object` for the report; the measurements live in memory but never make it to S3. The second most common miss is a layout name typo: `orders_fact_a` is one underscore between `fact` and `a`, not a dash.

</details>

<details><summary>Hint 2 (direction)</summary>

Two writes in this task. First, loop over `QUERY_SHAPES.items()` and call `measure_one(shape, layout, sql)` for each `(shape, layout)` pair, appending the dict to `MEASUREMENTS`. Second, `s3.put_object(Bucket=BUCKET_NAME, Key='report/measurements.json', Body=json.dumps(MEASUREMENTS, indent=2).encode('utf-8'))` to persist. The `measure_one` helper is already provided; it runs the query, captures `Duration` from `describe_statement`, then queries `STL_QUERY` + `STL_SCAN` to sum scan bytes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
MEASUREMENTS.clear()
for shape, layouts_sql in QUERY_SHAPES.items():
    for layout, sql in layouts_sql.items():
        m = measure_one(shape, layout, sql)
        MEASUREMENTS.append(m)
        print(f"{shape:>11} on {layout:>14}: scan_bytes={m['scanned_bytes']:>10} wall_ms={m['wall_ms']:>5}")

report_bytes = json.dumps(MEASUREMENTS, indent=2).encode("utf-8")
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="report/measurements.json",
    Body=report_bytes,
)
```

If a measurement returns `scanned_bytes=0` with a `note` field, STL_QUERY did not find the query by text match; this happens when the SQL has unusual whitespace. Re-run the cell; the second pass usually catches up because STL_QUERY is a few seconds behind real time. If the q_join sanity bound fails, double-check that `FACT_A` was created with `DISTKEY(customer_id)` and not just `DISTSTYLE KEY` without the key column. If the q_lookup sanity bound fails, confirm `DIM_C` is `DISTSTYLE ALL` via `SELECT diststyle FROM SVV_TABLE_INFO WHERE "table" = 'tier_dim_c';`.

</details>

**Wallet check.** Cluster meter still running at $0.25 per hour. Each query against the cluster is free; you pay for node-hours, not queries. STL_QUERY and STL_SCAN reads are queries against system tables, also free. Three S3 PUT requests for the seed + one for the report; well inside the free-tier request quota. Total session cost: around 4 to 6 cents at this point. About 15 minutes of cluster wall-clock spent.

## Task 4: Write the decision markdown

The lab does not pick your distribution for you. It gives you nine measurements and a constraint scenario, and you write a one-page decision the team can defend in a code review.

**Constraint scenario.** The analytics team runs the wide-scan aggregate 50x per day. The high-cardinality join runs 5x per day. The single-row lookup runs 200x per day. The fact tables are append-only and grow at about 1M new orders per week.

Write `decision.md` with four H2 sections:

- `## Layouts measured`: a bullet list of the three layouts and their DISTSTYLE/SORTKEY choices.
- `## Constraint scenario`: the workload mix above, plus growth.
- `## Chosen layout`: name your pick (one of `orders_fact_a` (KEY+SORTKEY), `orders_fact_b` (EVEN+SORTKEY), or a hybrid like "KEY+SORTKEY on the fact and ALL on the dim"). The validator captures this string verbatim; it does NOT grade your choice.
- `## Justification`: 2 to 4 sentences citing the measurement numbers from `measurements.json` (e.g., "KEY+SORTKEY scanned X MB on the join vs Y MB for EVEN; the wide-scan workload at 50x/day amortizes scan-bytes more than the join at 5x/day").

Save to `s3://{BUCKET}/report/decision.md`.

In [ ]:
# NBVAL_SKIP
# Task 4: Compose the decision markdown using the measurements captured in
# Task 3, and persist to s3://.../report/decision.md. The validator parses
# the H2 sections; it does NOT score the choice itself.

# Pull the measurements straight from the in-memory list (Task 3 already
# populated MEASUREMENTS and uploaded the JSON). Build a small lookup so
# the justification can quote scan-byte numbers.
m_by_pair = {(m["shape"], m["layout"]): m for m in MEASUREMENTS}

def fmt_kb(n):
    if n <= 0:
        return "0 B"
    if n < 1024:
        return f"{n} B"
    if n < 1024 * 1024:
        return f"{n/1024:.1f} KB"
    if n < 1024 * 1024 * 1024:
        return f"{n/(1024*1024):.1f} MB"
    return f"{n/(1024*1024*1024):.2f} GB"

# The student writes the four H2 sections. Pre-populated text shows the
# shape; the validator only checks H2 headers + that the chosen layout
# string appears in the "Chosen layout" section.
# YOUR CODE: assign DECISION_MARKDOWN to a string containing four H2 sections:
# '## Layouts measured', '## Constraint scenario', '## Chosen layout',
# '## Justification'. Cite at least one measurement number from MEASUREMENTS
# in the justification. Save to module-level DECISION_MARKDOWN.
DECISION_MARKDOWN = f"""# Redshift layout decision

## Layouts measured

- `{FACT_A}` with `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`. {ORDERS_ROWS_PER_LAYOUT:,} rows.
- `{FACT_B}` with `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)`. {ORDERS_ROWS_PER_LAYOUT:,} rows.
- `{DIM_C}` with `DISTSTYLE ALL`. {TIER_ROWS} rows.

## Constraint scenario

Analytics team workload mix:
- Wide-scan aggregate: 50x per day.
- High-cardinality join on customer_id: 5x per day.
- Single-row lookup: 200x per day.
- Fact growth: ~1M new orders per week, append-only.

## Chosen layout

`{FACT_A}` (KEY+SORTKEY on the fact) plus `{DIM_C}` (ALL on the small dim).

## Justification

On the high-cardinality join, KEY co-location wins: `{FACT_A}` scanned
{fmt_kb(m_by_pair[('q_join', FACT_A)]['scanned_bytes'])} vs
{fmt_kb(m_by_pair[('q_join', FACT_B)]['scanned_bytes'])} for `{FACT_B}`
with EVEN redistribution. On the wide-scan aggregate, the sortkey on
`order_date` does most of the work for both KEY and EVEN, so the choice
comes down to the 5x-daily join shape, which KEY wins. The dim is small
and read 200x daily; ALL distribution scans
{fmt_kb(m_by_pair[('q_lookup', DIM_C)]['scanned_bytes'])} vs
{fmt_kb(m_by_pair[('q_lookup', FACT_A)]['scanned_bytes'])} on the fact,
and never broadcasts. The 50x-daily wide-scan workload does NOT change
the pick because both KEY and EVEN scan a similar number of bytes when
sortkey on `order_date` is in place; the join shape is the tiebreaker.
"""

# Persist to S3.
# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key='report/decision.md',
# Body=DECISION_MARKDOWN.encode('utf-8')).
print(f"decision.md uploaded to s3://{BUCKET_NAME}/report/decision.md ({len(DECISION_MARKDOWN)} chars)")
print()
print("Captured comparison line for the companion panel:")
ja = fmt_kb(m_by_pair[('q_join', FACT_A)]['scanned_bytes'])
jb = fmt_kb(m_by_pair[('q_join', FACT_B)]['scanned_bytes'])
lc = fmt_kb(m_by_pair[('q_lookup', DIM_C)]['scanned_bytes'])
print(f"  KEY+SORTKEY scanned {ja} on the join; EVEN+SORTKEY scanned {jb}; ALL on dim scanned {lc} on the lookup.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: decision.md exists at s3://.../report/decision.md, contains
# all four required H2 sections, and the 'Chosen layout' section is
# non-empty. The validator does NOT score the choice.

def checkpoint_4(session):
    try:
        s3_c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_c.get_object(Bucket=BUCKET_NAME, Key="report/decision.md")
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"s3://{BUCKET_NAME}/report/decision.md does not exist. "
                        f"Task 4 must upload the decision markdown."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        body = obj["Body"].read().decode("utf-8")
        required_sections = [
            "## Layouts measured",
            "## Constraint scenario",
            "## Chosen layout",
            "## Justification",
        ]
        for s in required_sections:
            if s not in body:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"decision.md is missing required H2 section {s!r}. "
                        f"All four H2 headers must be present verbatim."
                    ),
                )

        # The 'Chosen layout' section must be non-trivial: extract the body
        # between '## Chosen layout' and the next H2 header.
        idx = body.index("## Chosen layout")
        after = body[idx + len("## Chosen layout"):]
        next_h2 = after.find("\n## ")
        chosen_body = (after if next_h2 == -1 else after[:next_h2]).strip()
        if len(chosen_body) < 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "The 'Chosen layout' section is empty or under 5 chars. "
                    "Name the layout the team should adopt."
                ),
            )

        # Capture the comparison metric for the companion panel. Print it
        # verbatim from the measurements report so the student sees the
        # captured measurement on PASS.
        try:
            m_obj = s3_c.get_object(Bucket=BUCKET_NAME, Key="report/measurements.json")
            ms = json.loads(m_obj["Body"].read().decode("utf-8"))
            by_pair = {(m["shape"], m["layout"]): m for m in ms}
            ja = by_pair.get(("q_join", FACT_A), {}).get("scanned_bytes", 0)
            jb = by_pair.get(("q_join", FACT_B), {}).get("scanned_bytes", 0)
            lc = by_pair.get(("q_lookup", DIM_C), {}).get("scanned_bytes", 0)
            print(
                f"Comparison metric captured: KEY+SORTKEY scanned {ja} B on join; "
                f"EVEN+SORTKEY scanned {jb} B; ALL on dim scanned {lc} B on lookup."
            )
        except Exception:
            pass

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Checkpoint 4 walks two things: the markdown exists at `s3://.../report/decision.md`, and all four required H2 headers (`## Layouts measured`, `## Constraint scenario`, `## Chosen layout`, `## Justification`) appear verbatim with a non-empty `Chosen layout` section. The most common miss is forgetting to call `s3.put_object`; the string lives in the Python variable but never reaches S3.

</details>

<details><summary>Hint 2 (direction)</summary>

Assign `DECISION_MARKDOWN` to a Python string with the four H2 headers in order, each followed by content. Then call `s3.put_object(Bucket=BUCKET_NAME, Key='report/decision.md', Body=DECISION_MARKDOWN.encode('utf-8'))`. The pre-populated string in the task cell already shows the shape; you can adjust the `Chosen layout` and `Justification` content to reflect your own reasoning, or leave them as-is. The validator does NOT score the choice itself.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
DECISION_MARKDOWN = f"""# Redshift layout decision

## Layouts measured

- `{FACT_A}` with `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`. {ORDERS_ROWS_PER_LAYOUT:,} rows.
- `{FACT_B}` with `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)`. {ORDERS_ROWS_PER_LAYOUT:,} rows.
- `{DIM_C}` with `DISTSTYLE ALL`. {TIER_ROWS} rows.

## Constraint scenario

Wide-scan aggregate 50x/day, high-cardinality join 5x/day, single-row lookup 200x/day. Fact growth ~1M new orders/week.

## Chosen layout

`{FACT_A}` (KEY+SORTKEY on the fact) plus `{DIM_C}` (ALL on the small dim).

## Justification

On the high-cardinality join, KEY co-location wins: FACT_A scanned <X> bytes vs FACT_B's <Y> bytes with EVEN redistribution. The wide-scan workload at 50x/day uses the sortkey on order_date equally on both layouts, so the join shape (5x/day) is the tiebreaker. The dim is small and read 200x/day; ALL distribution avoids broadcast and scans only one slice's copy.
\"\"\"

s3.put_object(
    Bucket=BUCKET_NAME,
    Key="report/decision.md",
    Body=DECISION_MARKDOWN.encode("utf-8"),
)
```

</details>

**Wallet check.** Cluster meter still running at $0.25 per hour. The decision markdown is a single S3 PUT; the cost is one tiny request inside the free-tier quota. Total session cost: 6 to 8 cents at this point. About 20 to 25 minutes of cluster wall-clock spent. The cleanup cell below is what stops the meter; do not close this tab without running it.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in critical-first,
# reverse-creation order per RESOURCE-SAFETY-SPEC Rule 2. The cluster and
# the safety-net pair (EventBridge rule + Lambda) go FIRST because they are
# the hourly-billed resources. Then the two IAM roles, the SG, and the
# bucket. labrun-checks 0.7.0 ships AWS adapters for all seven types, so
# run_cleanup against the default adapter handles everything.
#
# Two-step ordering for the EventBridge rule is handled inside the labrun-checks
# eventbridge_rule adapter: remove-targets first, then delete-rule.
#
# Empty the S3 bucket BEFORE run_cleanup because the s3_bucket adapter only
# handles a single page of objects on its own.

print("Emptying the S3 bucket before teardown...")
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_failed = sum(1 for r in CLEANUP_MANIFEST if r.critical and r.id in failed_ids)
standard_failed = len(failed_ids) - critical_failed
critical_destroyed = critical_total - critical_failed
standard_destroyed = standard_total - standard_failed
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.30 to $0.50.** The Redshift `dc2.large` node-hour is the only line item that materially registered. At $0.25 per hour, a 75-minute session runs $0.30 to $0.35 if cleanup ran clean. Worst case (kernel died, atexit ran late, the 2-hour EventBridge safety net fired) is $0.50. The S3 seed Parquet, IAM, SG, Lambda, and EventBridge rule were all free at this volume. The catastrophic case (cleanup skipped, atexit failed, safety-net Lambda failed) would require three independent failures and is the reason the $20-per-month AWS billing alert exists. Check your AWS Billing console in 24 hours to confirm the exact number; if you ran cleanup and see this message with zero errors above, your bill should match this estimate closely.

## These are not graded. They are for you.

Three questions worth sitting with before your next data-warehouse interview.

1. Walk through what each distribution choice physically changes about how Redshift stores and reads the table. `DISTKEY(customer_id)` hashes rows across slices by customer_id; `DISTSTYLE EVEN` round-robins; `DISTSTYLE ALL` copies the entire table to every slice. Which redistribution operations (`DS_BCAST_INNER`, `DS_DIST_BOTH`, `DS_DIST_NONE`) does each layout produce on the high-cardinality join versus the wide-scan? Which combinations are degenerate (you would never pick them in production) and why?

2. The constraint scenario gave you a daily workload split of 50x wide-scan, 5x join, 200x lookup. If the team's product owner showed up tomorrow and changed it to 5x wide-scan, 50x join, 200x lookup, would your `Chosen layout` flip? What measurement number in `measurements.json` would you cite when the team pushes back? Sketch the same justification paragraph under the new constraint and identify the line that flips the decision.

3. The lab pinned the dim at `DISTSTYLE ALL` because 12 rows is tiny. At what dim size does ALL stop making sense, and why? Walk through the storage cost of ALL on a 12-row dim, a 12,000-row dim, and a 12-million-row dim across a 4-slice and a 32-slice cluster. Which dim size is the breakpoint where you would switch to KEY or EVEN, and what other property of the dim (cardinality, update frequency, join shape) would push the breakpoint in either direction?

## Exam-Style Practice

**Q1.** Your analytics team runs three queries against a 200M-row order fact: a high-cardinality join on `customer_id` (5x/day), a wide-scan aggregate over `order_date` ranges (50x/day), and a single-row lookup against a 12-row tier dim (200x/day). Which Redshift layout wins overall?

A) `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)` on the fact, `DISTSTYLE ALL` on the tier dim.

B) `DISTSTYLE ALL` on every table because the analyst queries are all simple shapes.

C) `DISTKEY(order_date) COMPOUND SORTKEY(customer_id)` on the fact, `DISTSTYLE EVEN` on the dim.

D) `DISTSTYLE EVEN COMPOUND SORTKEY(order_date)` on the fact and `DISTSTYLE EVEN` on the dim.

<details><summary>Show answer</summary>

**A**.

- A) Right because the wide-scan aggregate uses the `order_date` sortkey equally on KEY and EVEN, so the 50x daily wide-scan workload does NOT differentiate the layouts; the join shape (5x/day) is what differentiates, and KEY+SORTKEY wins by co-locating customer rows on a single slice. The dim is small and read 200x/day; ALL avoids broadcast and never redistributes. This is the canonical "KEY on fact + ALL on dim" pattern.
- B) Wrong because `DISTSTYLE ALL` on a 200M-row fact copies the entire fact to every slice; storage explodes and writes get expensive. ALL is only sensible for small static dims.
- C) Wrong because distributing on `order_date` collapses the fact onto a small number of slices (one per distinct date hash), which kills parallelism on every query. Sortkey on `customer_id` does not help the wide-scan workload at all.
- D) Wrong because EVEN forces redistribution on every join on `customer_id`. The wide-scan aggregate would tie with option A but the join (even at 5x/day) is the tiebreaker, and EVEN loses it. EVEN on the dim is harmless but a missed opportunity; ALL is strictly better for a 12-row dim.

</details>

**Q2.** A data engineer creates a new Redshift table with `DISTKEY(order_id) COMPOUND SORTKEY(order_date)` and runs the query `SELECT customer_id, sum(amount_cents) FROM orders GROUP BY customer_id;`. The EXPLAIN plan shows `DS_DIST_BOTH`. What is the underlying reason and the cleanest fix?

A) The sortkey is on `order_date` instead of `customer_id`; rebuild the table with `COMPOUND SORTKEY(customer_id)` so the aggregation can sort-and-stream.

B) The distkey is on `order_id`, which is high-cardinality and unrelated to the GROUP BY column. The aggregation forces both sides to redistribute by `customer_id`; rebuild the table with `DISTKEY(customer_id)` so the GROUP BY stays local.

C) `DS_DIST_BOTH` is normal for any aggregation; no fix is needed.

D) The table needs `VACUUM REINDEX` to populate the slice statistics; the redistribution will go away after the next vacuum.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because sortkey controls within-slice ordering, not redistribution. The redistribution happens at the slice boundary, not at the sortkey scan; a sortkey change does not fix `DS_DIST_BOTH`.
- B) Right because `DS_DIST_BOTH` means both sides of the operation are being redistributed across slices. The distkey is on `order_id` but the GROUP BY is on `customer_id`, so rows for the same customer live on different slices; aggregation forces a redistribution. The fix is to make the distkey match the join/group-by column.
- C) Wrong because `DS_DIST_BOTH` is specifically the high-cost case the EXPLAIN plan warns about. `DS_DIST_NONE` is the goal for local operations.
- D) Wrong because `VACUUM REINDEX` rebuilds sortkey order; it does nothing to distribution. The redistribution is structural, not statistical.

</details>

**Q3.** A team is moving from `dc2.large` provisioned Redshift to `RA3.xlplus`. Their existing tables use `DISTKEY(customer_id) COMPOUND SORTKEY(order_date)`. Which Redshift feature do they gain that materially changes the distribution-tuning workflow on the new cluster?

A) `ALTER TABLE ALTER DISTKEY` and `ALTER TABLE ALTER SORTKEY` become available, so distribution choices stop being permanent and can be re-tuned in place without DROP/CREATE/COPY.

B) `DISTSTYLE ALL` no longer exists on RA3; teams must rewrite small-dim tables with KEY or EVEN.

C) RA3 automatically picks the distkey for every table; the manual choice is removed.

D) `STL_QUERY` and `STL_SCAN` are removed from RA3 in favor of the redshift-data API.

<details><summary>Show answer</summary>

**A**.

- A) Right because RA3 nodes support in-place `ALTER TABLE ALTER DISTKEY/ALTER SORTKEY` (and AUTO distkey/sortkey at table create), so the painful DROP+CREATE+COPY cycle this lab walked through stops being mandatory. On dc2/ds2 you had to bake the choice in at create time and recreate to change it; on RA3 you can adjust as the workload shifts.
- B) Wrong because `DISTSTYLE ALL` is supported on every node type including RA3. The small-dim pattern this lab teaches translates directly.
- C) Wrong because RA3 exposes `DISTSTYLE AUTO` as an optional default but does not force it; explicit KEY/EVEN/ALL are still the right call when the workload is known.
- D) Wrong because `STL_QUERY` and `STL_SCAN` are present on all provisioned node types. The redshift-data API is a transport; STL views are the storage.

</details>